# UK Logistics & Road Freight Intelligence Dashboard

## Notebook 02: Data Cleaning and Preparation

This notebook cleans and prepares the downloaded Department for Transport road freight and road traffic datasets.

The purpose of this notebook is to:

- Load raw CSV and ODS files
- Standardise column names
- Clean road traffic CSV datasets
- Extract useful ODS road freight tables
- Remove empty rows and columns
- Export cleaned datasets into the processed data folder
- Create a processed file inventory for the next analysis stage

The cleaned outputs will support exploratory analysis, SQL database creation and Power BI reporting.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

In [2]:
# Define project paths

raw_data_path = Path("../data/raw")
processed_data_path = Path("../data/processed")

processed_data_path.mkdir(parents=True, exist_ok=True)

print("Raw data folder exists:", raw_data_path.exists())
print("Processed data folder exists:", processed_data_path.exists())

Raw data folder exists: True
Processed data folder exists: True


In [3]:
def clean_column_name(column_name):
    """
    Convert a column name into clean snake_case format.
    """
    column_name = str(column_name).strip().lower()
    column_name = re.sub(r"[^a-z0-9]+", "_", column_name)
    column_name = re.sub(r"_+", "_", column_name)
    column_name = column_name.strip("_")
    return column_name


def standardise_columns(df):
    """
    Standardise all dataframe column names.
    """
    df = df.copy()
    df.columns = [clean_column_name(col) for col in df.columns]
    return df


def clean_basic_dataframe(df):
    """
    Basic cleaning applied to raw datasets.
    """
    df = df.copy()

    # Drop completely empty rows and columns
    df = df.dropna(how="all")
    df = df.dropna(axis=1, how="all")

    # Standardise column names
    df = standardise_columns(df)

    return df


def safe_file_stem(file_path):
    """
    Create safe file name stem.
    """
    return clean_column_name(file_path.stem)

In [4]:
# List raw files

raw_files = sorted([file for file in raw_data_path.iterdir() if file.is_file()])

raw_file_summary = pd.DataFrame([
    {
        "file_name": file.name,
        "extension": file.suffix.lower(),
        "size_mb": round(file.stat().st_size / (1024 * 1024), 2)
    }
    for file in raw_files
])

raw_file_summary

,file_name,extension,size_mb
0,.gitkeep,,0.00
1,local_authority_traffic.csv,.csv,0.43
2,region_traffic_by_road_type.csv,.csv,0.10
3,region_traffic_by_vehicle_type.csv,.csv,0.04
4,rfs0101.ods,.ods,0.01
5,rfs0121.ods,.ods,0.01
6,rfs0122.ods,.ods,0.01
7,rfs0125.ods,.ods,0.01


## Clean CSV Road Traffic Files

This section reads all CSV files from the raw data folder, standardises column names and exports cleaned versions into the processed data folder.

In [5]:
# Clean CSV files

csv_files = [file for file in raw_files if file.suffix.lower() == ".csv"]

cleaned_csv_outputs = []

print("CSV files found:", len(csv_files))

for file in csv_files:
    print("\nProcessing:", file.name)
    
    try:
        df = pd.read_csv(file, low_memory=False)
        df_clean = clean_basic_dataframe(df)
        
        output_name = f"cleaned_{safe_file_stem(file)}.csv"
        output_path = processed_data_path / output_name
        
        df_clean.to_csv(output_path, index=False)
        
        cleaned_csv_outputs.append({
            "source_file": file.name,
            "output_file": output_name,
            "rows": df_clean.shape[0],
            "columns": df_clean.shape[1]
        })
        
        print("Exported:", output_name)
        print("Rows:", df_clean.shape[0], "| Columns:", df_clean.shape[1])
        
    except Exception as error:
        print("Could not process:", file.name)
        print("Error:", error)

cleaned_csv_outputs_df = pd.DataFrame(cleaned_csv_outputs)
cleaned_csv_outputs_df

CSV files found: 3

Processing: local_authority_traffic.csv
Exported: cleaned_local_authority_traffic.csv
Rows: 6560 | Columns: 8

Processing: region_traffic_by_road_type.csv
Exported: cleaned_region_traffic_by_road_type.csv
Rows: 1623 | Columns: 9

Processing: region_traffic_by_vehicle_type.csv
Exported: cleaned_region_traffic_by_vehicle_type.csv
Rows: 352 | Columns: 13


,source_file,output_file,rows,columns
0,local_authority_traffic.csv,cleaned_local_authority_traffic.csv,6560,8
1,region_traffic_by_road_type.csv,cleaned_region_traffic_by_road_type.csv,1623,9
2,region_traffic_by_vehicle_type.csv,cleaned_region_traffic_by_vehicle_type.csv,352,13


## Extract ODS Road Freight Tables

This section attempts to read useful sheets from the ODS road freight workbooks.

ODS files often contain title rows and notes, so this first cleaning stage extracts readable sheets, removes empty rows and columns, standardises columns and exports local processed CSV files.

In [6]:
# Helper function to detect a likely header row in an ODS sheet

def detect_header_row(raw_df, max_scan_rows=30):
    """
    Try to detect the most likely header row in a raw ODS sheet.
    Looks for rows with useful keywords or many non-empty cells.
    """
    keyword_patterns = [
        "year",
        "date",
        "region",
        "country",
        "vehicle",
        "goods",
        "tonne",
        "kilometre",
        "kilometer",
        "traffic",
        "road"
    ]
    
    best_row = 0
    best_score = -1
    
    scan_limit = min(max_scan_rows, len(raw_df))
    
    for idx in range(scan_limit):
        row_values = raw_df.iloc[idx].astype(str).str.lower().tolist()
        non_empty_count = sum(
            value not in ["nan", "none", ""] for value in row_values
        )
        keyword_count = sum(
            any(keyword in value for keyword in keyword_patterns)
            for value in row_values
        )
        
        score = non_empty_count + (keyword_count * 3)
        
        if score > best_score:
            best_score = score
            best_row = idx
    
    return best_row

In [7]:
# Clean ODS files

ods_files = [file for file in raw_files if file.suffix.lower() == ".ods"]

cleaned_ods_outputs = []

print("ODS files found:", len(ods_files))

for file in ods_files:
    print("\n" + "=" * 80)
    print("Processing ODS file:", file.name)
    print("=" * 80)
    
    try:
        excel_file = pd.ExcelFile(file, engine="odf")
        
        for sheet_name in excel_file.sheet_names:
            print("Sheet:", sheet_name)
            
            try:
                raw_sheet = pd.read_excel(
                    file,
                    sheet_name=sheet_name,
                    engine="odf",
                    header=None
                )
                
                raw_sheet = raw_sheet.dropna(how="all")
                raw_sheet = raw_sheet.dropna(axis=1, how="all")
                
                if raw_sheet.empty:
                    print("Skipped empty sheet")
                    continue
                
                header_row = detect_header_row(raw_sheet)
                
                df_sheet = pd.read_excel(
                    file,
                    sheet_name=sheet_name,
                    engine="odf",
                    header=header_row
                )
                
                df_sheet = clean_basic_dataframe(df_sheet)
                
                # Only export sheets with meaningful size
                if df_sheet.shape[0] < 2 or df_sheet.shape[1] < 2:
                    print("Skipped small/non-data sheet")
                    continue
                
                safe_sheet_name = clean_column_name(sheet_name)
                output_name = f"cleaned_{safe_file_stem(file)}_{safe_sheet_name}.csv"
                output_path = processed_data_path / output_name
                
                df_sheet.to_csv(output_path, index=False)
                
                cleaned_ods_outputs.append({
                    "source_file": file.name,
                    "sheet_name": sheet_name,
                    "output_file": output_name,
                    "rows": df_sheet.shape[0],
                    "columns": df_sheet.shape[1],
                    "detected_header_row": header_row
                })
                
                print("Exported:", output_name)
                print("Rows:", df_sheet.shape[0], "| Columns:", df_sheet.shape[1])
                
            except Exception as sheet_error:
                print("Could not process sheet:", sheet_name)
                print("Error:", sheet_error)
        
    except Exception as file_error:
        print("Could not open ODS file:", file.name)
        print("Error:", file_error)

cleaned_ods_outputs_df = pd.DataFrame(cleaned_ods_outputs)
cleaned_ods_outputs_df

ODS files found: 4

Processing ODS file: rfs0101.ods
Sheet: Cover_sheet
Skipped small/non-data sheet
Sheet: Table_of_contents
Could not process sheet: Table_of_contents
Error: argument of type 'float' is not iterable
Sheet: RFS0101_Quarterly
Could not process sheet: RFS0101_Quarterly
Error: argument of type 'float' is not iterable
Sheet: RFS0101_Annual
Could not process sheet: RFS0101_Annual
Error: argument of type 'float' is not iterable
Sheet: RFS0101_Notes
Could not process sheet: RFS0101_Notes
Error: argument of type 'float' is not iterable

Processing ODS file: rfs0121.ods
Sheet: Cover_sheet
Skipped small/non-data sheet
Sheet: Table_of_contents
Could not process sheet: Table_of_contents
Error: argument of type 'float' is not iterable
Sheet: RFS0121_GoodsLifted
Could not process sheet: RFS0121_GoodsLifted
Error: argument of type 'float' is not iterable
Sheet: RFS0121_GoodsMoved
Could not process sheet: RFS0121_GoodsMoved
Error: argument of type 'float' is not iterable
Sheet: RFS012

""


In [8]:
# Create processed file inventory

processed_files = sorted([
    file for file in processed_data_path.iterdir()
    if file.is_file() and file.suffix.lower() == ".csv"
])

processed_inventory = []

for file in processed_files:
    try:
        df_preview = pd.read_csv(file, nrows=5, low_memory=False)
        row_count = sum(1 for _ in open(file, encoding="utf-8", errors="ignore")) - 1
        
        processed_inventory.append({
            "file_name": file.name,
            "file_size_mb": round(file.stat().st_size / (1024 * 1024), 2),
            "estimated_rows": row_count,
            "columns": df_preview.shape[1],
            "column_names": ", ".join(df_preview.columns.tolist()[:10])
        })
        
    except Exception as error:
        processed_inventory.append({
            "file_name": file.name,
            "file_size_mb": round(file.stat().st_size / (1024 * 1024), 2),
            "estimated_rows": "error",
            "columns": "error",
            "column_names": str(error)
        })

processed_inventory_df = pd.DataFrame(processed_inventory)

processed_inventory_df

,file_name,file_size_mb,estimated_rows,columns,column_names
0,cleaned_local_authority_traffic.csv,0.44,6560,8,"local_authority_id, local_authority_name, loca..."
1,cleaned_region_traffic_by_road_type.csv,0.10,1623,9,"year, region_id, region_name, region_ons_code,..."
2,cleaned_region_traffic_by_vehicle_type.csv,0.05,352,13,"year, region_id, region_name, region_ons_code,..."


In [9]:
# Export inventory files

raw_file_summary.to_csv(processed_data_path / "raw_file_inventory.csv", index=False)
processed_inventory_df.to_csv(processed_data_path / "processed_file_inventory.csv", index=False)

print("Inventory files exported successfully.")
print("Raw inventory:", (processed_data_path / "raw_file_inventory.csv").exists())
print("Processed inventory:", (processed_data_path / "processed_file_inventory.csv").exists())

Inventory files exported successfully.
Raw inventory: True
Processed inventory: True


## Cleaning Summary

This notebook completed the first cleaning stage for the UK logistics and road freight project.

Completed actions:

- Listed all raw Department for Transport files
- Cleaned CSV road traffic files
- Extracted and cleaned ODS road freight workbook sheets
- Standardised column names
- Removed empty rows and columns
- Exported cleaned local CSV files into the processed data folder
- Created raw and processed file inventories

The next notebook will review the cleaned outputs and select the most useful tables for exploratory analysis, SQL reporting and Power BI dashboard development.